In [ ]:
import cv2
import numpy as np
import pandas as pd
import tensorflow
from tensorflow.keras.preprocessing.image import img_to_array

2026-06-09 20:26:11.901187: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-09 20:26:12.072380: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-09 20:26:12.072923: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-09 20:26:13.266029: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
from tensorflow.keras.models import load_model
import operator

In [3]:
# Nome dos modelos que serão comparados
model_archives = ["modelo_01_expressoes.h5", "modelo_02_expressoes.h5", "modelo_03_expressoes.h5", "modelo_04_expressoes.h5", "modelo_05_expressoes.h5"]
# dicionario dos modelos
models = {}

In [4]:
# carregando os dados de teste
x_test = np.load('../Material/Material/mod_xtest.npy')
y_test = np.load('../Material/Material/mod_ytest.npy')

In [5]:
for model in model_archives:
    # carrega o modelo
    loaded_model = load_model('../Material/Material/' + model)

    # pega as estatisticas do modelo treinado
    scores = loaded_model.evaluate(np.array(x_test), np.array(y_test), batch_size=64)

    # mostra as stats
    print("---"+ str(model) +"---")
    print("Perda/Loss: " + str(scores[0]))
    print("Acurácia: " + str(scores[1]))
    models[model] = str(scores[1])
    print("\n")

57/57 [==============================] - 17s 286ms/step - loss: 1.0704 - accuracy: 0.6392
---modelo_01_expressoes.h5---
Perda/Loss: 1.070416808128357
Acurácia: 0.6391752362251282


57/57 [==============================] - 6s 104ms/step - loss: 1.0119 - accuracy: 0.6411
---modelo_02_expressoes.h5---
Perda/Loss: 1.0118968486785889
Acurácia: 0.6411256790161133


57/57 [==============================] - 16s 283ms/step - loss: 1.0732 - accuracy: 0.6308
---modelo_03_expressoes.h5---
Perda/Loss: 1.0731697082519531
Acurácia: 0.6308164000511169


57/57 [==============================] - 26s 447ms/step - loss: 1.1691 - accuracy: 0.6160
---modelo_04_expressoes.h5---
Perda/Loss: 1.169052004814148
Acurácia: 0.61604905128479


57/57 [==============================] - 17s 288ms/step - loss: 1.8206 - accuracy: 0.2455
---modelo_05_expressoes.h5---
Perda/Loss: 1.820610761642456
Acurácia: 0.2454722821712494




In [6]:
# ordena os modelos
order = sorted(models.items(), key=operator.itemgetter(1), reverse=True)
print(order)

[('modelo_02_expressoes.h5', '0.6411256790161133'), ('modelo_01_expressoes.h5', '0.6391752362251282'), ('modelo_03_expressoes.h5', '0.6308164000511169'), ('modelo_04_expressoes.h5', '0.61604905128479'), ('modelo_05_expressoes.h5', '0.2454722821712494')]


In [7]:
# melhor modelo
order[0]

('modelo_02_expressoes.h5', '0.6411256790161133')

In [ ]:
# carrega e mostra a imagem
image = cv2.imread('../Material/Material/testes/teste_gabriel.png')
cv2.imshow('Imagem', image)
cv2.waitKey(1000)
cv2.destroyAllWindows()

In [ ]:
# carregando o detector de faces e o modelo
cascade_faces = '../Material/Material/haarcascade_frontalface_default.xml'
model_path = '../Material/Material/' + str(order[0][0])
face_detection = cv2.CascadeClassifier(cascade_faces)
classifier = load_model(model_path, compile=False)
expressions = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]

In [13]:
# copia da imagem
original = image.copy()
# detectando a face
faces = face_detection.detectMultiScale(original,scaleFactor=1.1,minNeighbors=3,minSize=(20,20))
# em escala de cinza
gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)

In [ ]:
# fazendo o o recorte da area de interesse e desenhando os retangulos
if len(faces) > 0:
    for (fX, fY, fW, fH) in faces:
      roi = gray[fY:fY + fH, fX:fX + fW]
      roi = cv2.resize(roi, (48, 48))
      roi = roi.astype("float") / 255.0
      roi = img_to_array(roi)
      roi = np.expand_dims(roi, axis=0)
      preds = classifier.predict(roi)[0]
      print(preds)
      emotion_probability = np.max(preds)
      label = expressions[preds.argmax()]
      cv2.putText(original, label, (fX, fY - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2, cv2.LINE_AA)
      cv2.rectangle(original, (fX, fY), (fX + fW, fY + fH),(0, 0, 255), 2)
else:
    print('Nenhuma face detectada')

1/1 [==============================] - 0s 317ms/step
[1.2285652e-03 8.9724279e-05 7.5628120e-04 9.4787621e-01 2.9239706e-03
 5.5896194e-04 4.6566207e-02]


In [20]:
# imagem com o retangulo
cv2.imshow(' ', original)
cv2.waitKey(1000)

probabilities = np.ones((250, 300, 3), dtype="uint8") * 255
# Mostra gráfico apenas se detectou uma face
if len(faces) == 1:
  for (i, (emotion, prob)) in enumerate(zip(expressions, preds)):
      # Nome das emoções
      text = "{}: {:.2f}%".format(emotion, prob * 100)

      # cria as barrinhas de probabilidade para cada emocao
      w = int(prob * 300)
      cv2.rectangle(probabilities, (7, (i * 35) + 5),
      (w, (i * 35) + 35), (200, 250, 20), -1)
      cv2.putText(probabilities, text, (10, (i * 35) + 23),
      cv2.FONT_HERSHEY_SIMPLEX, 0.45,
      (0, 0, 0), 1, cv2.LINE_AA)

  cv2.imshow(' ', probabilities)
  cv2.waitKey(1000)

cv2.imwrite("capture.jpg",original)
cv2.destroyAllWindows()